In [19]:
import pandas as pd
import json

In [20]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [21]:
df_hr = df_raw[(df_raw['Key'] == 'heart_rate') & (df_raw['Tag'] == 'daily_report')].copy()

hr_data_list = [json.loads(val) for val in df_hr['Value']]
df_hr_normalized = pd.json_normalize(hr_data_list)

df_hr_result = df_hr[['Uid', 'Sid', 'Time']].reset_index(drop=True).join(df_hr_normalized)

df_hr_result = df_hr_result.drop(['abnormal_hr_count', 'latest_hr.bpm', 'latest_hr.time'], axis=1)

df_hr_result

,Uid,Sid,Time,avg_hr,max_hr,fat_burning_hr_zone_duration,extreme_hr_zone_duration,avg_rhr,anaerobic_hr_zone_duration,aerobic_hr_zone_duration,min_hr,warm_up_hr_zone_duration
0,8278074507,default,1736812800,103,161,13,0,71,2,3,66,45
1,8278074507,default,1736899200,67,151,54,0,51,0,3,48,35
2,8278074507,default,1736985600,54,107,0,0,50,0,0,43,20
3,8278074507,default,1737072000,57,99,0,0,0,0,0,47,1
4,8278074507,default,1737158400,50,79,0,0,0,0,0,43,0
...,...,...,...,...,...,...,...,...,...,...,...,...
305,8278074507,default,1775088000,56,129,6,0,57,0,0,46,11
306,8278074507,default,1775174400,54,105,0,0,58,0,0,42,6
307,8278074507,default,1775260800,53,98,0,0,0,0,0,41,1
308,8278074507,default,1775347200,51,96,0,0,0,0,0,41,0


In [22]:
df_hr_result['Date'] = pd.to_datetime(df_hr_result['Time'], unit='s')

df_hr_result = df_hr_result.drop(columns=['Time', 'Uid', 'Sid'])

df_hr_result

,avg_hr,max_hr,fat_burning_hr_zone_duration,extreme_hr_zone_duration,avg_rhr,anaerobic_hr_zone_duration,aerobic_hr_zone_duration,min_hr,warm_up_hr_zone_duration,Date
0,103,161,13,0,71,2,3,66,45,2025-01-14
1,67,151,54,0,51,0,3,48,35,2025-01-15
2,54,107,0,0,50,0,0,43,20,2025-01-16
3,57,99,0,0,0,0,0,47,1,2025-01-17
4,50,79,0,0,0,0,0,43,0,2025-01-18
...,...,...,...,...,...,...,...,...,...,...
305,56,129,6,0,57,0,0,46,11,2026-04-02
306,54,105,0,0,58,0,0,42,6,2026-04-03
307,53,98,0,0,0,0,0,41,1,2026-04-04
308,51,96,0,0,0,0,0,41,0,2026-04-05


In [23]:
import os

output_dir = '../data/processed_data'
output_path = os.path.join(output_dir, 'miband_hr_processed.csv')
df_hr_result.to_csv(output_path, index=False)